# Faktorisering

In [1]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [2]:
heroes = pd.read_csv("../FASERIP.csv")
heroes.head()


,character,F,A,S,E,R,I,P,Type,Flight,Armour,Ranged,Alignment
0,Angel,20,30,10,30,10,6,10,Mutant,6,0,0,Hero
1,Aurora,20,20,6,20,4,10,4,Mutant,150,0,0,Hero
2,Beast,40,40,30,20,20,10,20,Mutant,0,0,0,Hero
3,Black Knight,30,20,10,10,10,6,6,Human,0,10,0,Hero
4,Box,30,30,75,75,20,10,10,Technological,40,50,0,Hero


In [3]:
# Select only the FASERIP columns
stats_only = heroes[['F', 'A', 'S', 'E', 'R', 'I', 'P']]

# Initialize the scaler
scaler = StandardScaler()

# "Fit and Transform" the data (this makes the mean 0 and variance 1)
stats_scaled = scaler.fit_transform(stats_only)

In [4]:
# We'll ask for all 7 components initially
pca = PCA(n_components=7)

# Run the math
pca.fit(stats_scaled)

,"n_components n_components: int, float or 'mle', default=NoneNumber of components to keep.if n_components is not set all components are kept:: n_components == min(n_samples, n_features)If ``n_components == 'mle'`` and ``svd_solver == 'full'``, Minka'sMLE is used to guess the dimension. Use of ``n_components == 'mle'``will interpret ``svd_solver == 'auto'`` as ``svd_solver == 'full'``.If ``0 < n_components < 1`` and ``svd_solver == 'full'``, select thenumber of components such that the amount of variance that needs to beexplained is greater than the percentage specified by n_components.If ``svd_solver == 'arpack'``, the number of components must bestrictly less than the minimum of n_features and n_samples.Hence, the None case results in:: n_components == min(n_samples, n_features) - 1",7
,"copy copy: bool, default=TrueIf False, data passed to fit are overwritten and runningfit(X).transform(X) will not yield the expected results,use fit_transform(X) instead.",True
,"whiten whiten: bool, default=FalseWhen True (False by default) the `components_` vectors are multipliedby the square root of n_samples and then divided by the singular valuesto ensure uncorrelated outputs with unit component-wise variances.Whitening will remove some information from the transformed signal(the relative variance scales of the components) but can sometimeimprove the predictive accuracy of the downstream estimators bymaking their data respect some hard-wired assumptions.",False
,"svd_solver svd_solver: {'auto', 'full', 'covariance_eigh', 'arpack', 'randomized'}, default='auto'""auto"" : The solver is selected by a default 'auto' policy is based on `X.shape` and `n_components`: if the input data has fewer than 1000 features and more than 10 times as many samples, then the ""covariance_eigh"" solver is used. Otherwise, if the input data is larger than 500x500 and the number of components to extract is lower than 80% of the smallest dimension of the data, then the more efficient ""randomized"" method is selected. Otherwise the exact ""full"" SVD is computed and optionally truncated afterwards.""full"" : Run exact full SVD calling the standard LAPACK solver via `scipy.linalg.svd` and select the components by postprocessing""covariance_eigh"" : Precompute the covariance matrix (on centered data), run a classical eigenvalue decomposition on the covariance matrix typically using LAPACK and select the components by postprocessing. This solver is very efficient for n_samples >> n_features and small n_features. It is, however, not tractable otherwise for large n_features (large memory footprint required to materialize the covariance matrix). Also note that compared to the ""full"" solver, this solver effectively doubles the condition number and is therefore less numerical stable (e.g. on input data with a large range of singular values).""arpack"" : Run SVD truncated to `n_components` calling ARPACK solver via `scipy.sparse.linalg.svds`. It requires strictly `0 < n_components < min(X.shape)`""randomized"" : Run randomized SVD by the method of Halko et al... versionadded:: 0.18.0.. versionchanged:: 1.5 Added the 'covariance_eigh' solver.",'auto'
,"tol tol: float, default=0.0Tolerance for singular values computed by svd_solver == 'arpack'.Must be of range [0.0, infinity)... versionadded:: 0.18.0",0.0
,"iterated_power iterated_power: int or 'auto', default='auto'Number of iterations for the power method computed bysvd_solver == 'randomized'.Must be of range [0, infinity)... versionadded:: 0.18.0",'auto'
,"n_oversamples n_oversamples: int, default=10This parameter is only relevant when `svd_solver=""randomized""`.It corresponds to the additional number of random vectors to sample therange of `X` so as to ensure proper conditioning. See:func:`~sklearn.utils.extmath.randomized_svd` for more details... versionadded:: 1.1",10
,"power_iteration_normalizer power_iteration_normalizer: {'auto', 'QR', 'LU', 'none'}, default='auto'Power iteration normalizer for randomized SVD 

In [5]:
# This prints the percentage of variance for each component
print(pca.explained_variance_ratio_)

[0.33120743 0.1907864  0.15925742 0.13065295 0.08194655 0.07154762
 0.03460162]


In [6]:
# Create a DataFrame of the loadings
loadings = pd.DataFrame(
    pca.components_.T, 
    columns=[f'PC{i+1}' for i in range(7)],
    index=['F', 'A', 'S', 'E', 'R', 'I', 'P']
)

print(loadings)

        PC1       PC2       PC3       PC4       PC5       PC6       PC7
F  0.510762  0.012178  0.171820 -0.102375 -0.222362  0.803061 -0.067966
A  0.333631  0.122343  0.551127  0.425685  0.603812 -0.100463  0.118708
S  0.536851 -0.249466 -0.271180 -0.070651 -0.129686 -0.265704  0.695421
E  0.551723 -0.054354 -0.304069  0.136363 -0.033608 -0.334865 -0.684347
R -0.056802  0.496059 -0.355312  0.716174 -0.258321  0.137985  0.160552
I  0.145932  0.589593  0.445035 -0.280518 -0.472210 -0.362005  0.017515
P  0.103357  0.570965 -0.420043 -0.439464  0.527547  0.110362  0.057134


In [7]:
# Create the coordinates
hero_scores = pca.transform(stats_scaled)

# Turn it into a nice DataFrame with the Hero names
hero_mapped = pd.DataFrame(hero_scores, columns=[f'PC{i+1}' for i in range(7)])
hero_mapped.insert(0, 'character', heroes['character'])

hero_mapped.head()



,character,PC1,PC2,PC3,PC4,PC5,PC6,PC7
0,Angel,-0.737911,-0.817428,0.469833,0.317715,0.517607,0.282464,-0.269113
1,Aurora,-1.228300,-1.124965,0.736876,-0.286325,0.048132,0.336401,-0.251259
2,Beast,0.273347,-0.033818,0.493285,0.664780,0.474395,1.174318,0.739517
3,Black Knight,-1.141464,-0.956024,0.589738,-0.021678,-0.052804,1.071215,0.199971
4,Box,1.986421,-0.983128,-0.917882,0.897851,-0.343771,-0.551529,0.282893


In [16]:
from sklearn.decomposition import FactorAnalysis

In [17]:
fa = FactorAnalysis(n_components=3, random_state=42)

In [18]:
fa.fit(stats_scaled)

,"n_components n_components: int, default=NoneDimensionality of latent space, the number of componentsof ``X`` that are obtained after ``transform``.If None, n_components is set to the number of features.",3
,"tol tol: float, default=1e-2Stopping tolerance for log-likelihood increase.",0.01
,"copy copy: bool, default=TrueWhether to make a copy of X. If ``False``, the input X gets overwrittenduring fitting.",True
,"max_iter max_iter: int, default=1000Maximum number of iterations.",1000
,"noise_variance_init noise_variance_init: array-like of shape (n_features,), default=NoneThe initial guess of the noise variance for each feature.If None, it defaults to np.ones(n_features).",None
,"svd_method svd_method: {'lapack', 'randomized'}, default='randomized'Which SVD method to use. If 'lapack' use standard SVD fromscipy.linalg, if 'randomized' use fast ``randomized_svd`` function.Defaults to 'randomized'. For most applications 'randomized' willbe sufficiently precise while providing significant speed gains.Accuracy can also be improved by setting higher values for`iterated_power`. If this is not sufficient, for maximum precisionyou should choose 'lapack'.",'randomized'
,"iterated_power iterated_power: int, default=3Number of iterations for the power method. 3 by default. Only usedif ``svd_method`` equals 'randomized'.",3
,"rotation rotation: {'varimax', 'quartimax'}, default=NoneIf not None, apply the indicated rotation. Currently, varimax andquartimax are implemented. See`""The varimax criterion for analytic rotation in factor analysis""`_H. F. Kaiser, 1958... versionadded:: 0.24",None
,"random_state random_state: int or RandomState instance, default=0Only used when ``svd_method`` equals 'randomized'. Pass an int forreproducible results across multiple function calls.See :term:`Glossary `.",42


In [19]:
loadings = pd.DataFrame(
    fa.components_.T,
    index=stats_only.columns,
    columns=["Factor1", "Factor2", "Factor3"]
)

print(loadings)

    Factor1   Factor2   Factor3
F  0.594498  0.283590 -0.290222
A  0.296217  0.352842 -0.163186
S  0.856971 -0.177226 -0.074261
E  0.861955  0.033426  0.186770
R -0.063115  0.237808  0.509069
I  0.043139  0.581105 -0.099672
P  0.107675  0.253656  0.247982
